# D02 · Byte stream to commands - COBS, parser, dispatch table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D02_byte_stream_to_commands_cobs_parser_dispatch_tab.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 70 min

**Skill Hub concepts:** Consistent Overhead Byte Stuffing; Streaming frame parser; Interrupt-driven ring buffer; Dispatch table; Idempotent command

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D02")

## Map: the receive path of a firmware console or telemetry link

```
 UART RX interrupt ──> ring buffer (ISR writes head, main reads tail) ──> parser_feed(byte)
     collects bytes until 0x00 ──> COBS decode ──> CRC-16 check ──> dispatch_table[cmd](args)
                                                                    └─> same seq as last? re-ACK, do not re-run
```

Wire format: `COBS( cmd | seq | args... | crc16_hi | crc16_lo ) | 0x00`

**COBS** removes every `0x00` from the payload, so `0x00` can only mean "end of frame". Each block starts with a code
byte = distance to the next zero (or 0xFF for 254 data bytes with no zero). Overhead: at most 1 byte per 254.

| Input | Encoded |
|---|---|
| `00` | `01 01` |
| `11 22 00 33` | `03 11 22 02 33` |
| `11 22 33 44` | `05 11 22 33 44` |
| `11 00 00 00` | `02 11 01 01 01` |

| Command | Args | Idempotent? | Why it matters |
|---|---|---|---|
| `0x01` SET_POS | int32 LE absolute | yes | running it twice gives the same state |
| `0x02` MOVE_BY | int32 LE relative | **no** | a retransmitted frame would move twice: dedupe by `seq` |
| `0x03` PING | none | yes | |

A dispatch table (`{cmd, min_len, handler}` array) replaces a growing `switch`, keeps handlers testable, and makes
"unknown command" one line.

In [ ]:
%%writefile proto.h
#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>
#define MAX_FRAME 64                      /* decoded bytes, CRC included */
typedef struct { uint8_t raw[MAX_FRAME + 2]; size_t len; bool overflow; } Parser;
size_t cobs_encode(const uint8_t *in, size_t n, uint8_t *out);
size_t cobs_decode(const uint8_t *in, size_t n, uint8_t *out);   /* 0 if malformed */
uint16_t crc16_ccitt(const uint8_t *p, size_t n);                  /* given */
size_t parser_feed(Parser *p, uint8_t byte, uint8_t *payload);    /* payload length when a valid frame ends, else 0 */

typedef struct { int32_t position; uint8_t last_seq; bool has_seq; int executed; } Device;
enum { RESP_OK = 0, RESP_DUP = 1, RESP_UNKNOWN = 2, RESP_BADLEN = 3 };
int handle_payload(Device *d, const uint8_t *pl, size_t n);         /* pl = cmd, seq, args */

In [ ]:
%%writefile crc.c
#include "proto.h"
uint16_t crc16_ccitt(const uint8_t *p, size_t n) {
    uint16_t c = 0xFFFF;
    while (n--) { c ^= (uint16_t)(*p++ << 8); for (int k = 0; k < 8; k++) c = (c & 0x8000) ? (uint16_t)((c << 1) ^ 0x1021) : (uint16_t)(c << 1); }
    return c;
}

## Task 1: COBS encode and decode

In [ ]:
%%writefile cobs.c
#include "proto.h"

size_t cobs_encode(const uint8_t *in, size_t n, uint8_t *out) {
    (void)in; (void)n; (void)out; return 0; /* TODO */
}

size_t cobs_decode(const uint8_t *in, size_t n, uint8_t *out) {
    (void)in; (void)n; (void)out; return 0; /* TODO */
}

## Task 2: streaming parser and dispatch table with sequence-number dedupe
`parser_feed` gets one byte at a time (as the main loop drains the ring buffer). It must survive garbage, bad CRCs and
frames longer than `MAX_FRAME`, and always resync on the next `0x00`. (Senders usually also put a `0x00` *before*
each frame, so noise that arrived first is flushed as a bad frame instead of corrupting the real one.)

In [ ]:
%%writefile proto.c
#include <string.h>
#include "proto.h"

size_t parser_feed(Parser *p, uint8_t byte, uint8_t *payload) {
    (void)p; (void)byte; (void)payload; return 0; /* TODO */
}

static int32_t rd_i32(const uint8_t *a) { return (int32_t)((uint32_t)a[0] | (uint32_t)a[1] << 8 | (uint32_t)a[2] << 16 | (uint32_t)a[3] << 24); }
static int do_set(Device *d, const uint8_t *a) { d->position = rd_i32(a); return RESP_OK; }
static int do_move(Device *d, const uint8_t *a) { d->position += rd_i32(a); return RESP_OK; }
static int do_ping(Device *d, const uint8_t *a) { (void)d; (void)a; return RESP_OK; }

typedef struct { uint8_t cmd; uint8_t args_len; int (*fn)(Device *, const uint8_t *); } Entry;
static const Entry table[] = { {0x01, 4, do_set}, {0x02, 4, do_move}, {0x03, 0, do_ping} };

int handle_payload(Device *d, const uint8_t *pl, size_t n) {
    (void)d; (void)pl; (void)n; (void)table; return RESP_UNKNOWN; /* TODO: look up pl[0] in table */
}

In [ ]:
lab.check("cobs")
lab.check("parser")
lab.check("dispatch")

## Checkpoint: Math
**M1 (Apply).** Maximum COBS-encoded size of a 100-byte message (without the 0x00 delimiter)?

**M2 (Analyze).** 115200 baud 8N1 arrives continuously; the main loop can stall 5 ms between drains. Smallest
power-of-two ring buffer (bytes) that never overflows?

**M3 (Apply).** COBS encoding of `11 00 22`, as a list of byte values?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** The host retransmits MOVE_BY +10 (seq 7) because the ACK was lost. The device already ran it.
Correct device behavior?
- a) run it again; the host knows best
- b) recognize seq 7 as a duplicate, skip execution, send the ACK again
- c) drop it silently

**K2 (Evaluate).** Compare COBS-with-delimiter framing against a length-prefixed format (`len | payload | crc`)
when a byte is lost mid-frame. Which resyncs faster and why?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d02.h`).
3. In `roboprep/checks/d02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()